<style>
.listening-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 17rem), 1fr)); gap: .65rem; }
.listening-card { padding: .75rem .85rem; background: #edf3ee; border-radius: .45rem; min-width: 0; }
.listening-card p { margin: 0 0 .55rem; }
audio { width: min(100%, 21rem); }
.data-table { width: 100%; font-variant-numeric: tabular-nums; }
.data-table th, .data-table td { padding: .42rem .5rem; border-bottom: 1px solid #dce2de; text-align: left; vertical-align: top; }
.data-table thead { background: #edf3ee; }
.table-wrap { overflow-x: auto; margin: .7rem 0; }
.data-table .value { display: block; white-space: nowrap; font-weight: 650; }
.data-table .rhythm { color: #126860; }
.data-table .kit { color: #aa4a26; }
.trial-set { margin: .4rem 0; border: 1px solid #cbd8d0; border-radius: .4rem; padding: .45rem .65rem; }
.trial-set summary { font-weight: 600; cursor: pointer; }
.trial-set button { color: #126860; border: 1px solid #92aaa3; border-radius: .3rem; padding: .15rem .4rem; background: #fff; text-align: left; }
.trial-set button:hover, .trial-set [aria-pressed="true"] { background: #e5eee9; }
</style>

In [ ]:
#| output: asis

from collections import Counter, defaultdict
from csv import DictReader
from itertools import combinations
from pathlib import Path
from statistics import median

import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

kit_names = {
    "virtuosity": "Virtuosity Drums",
    "black-pearl": "AVL Black Pearl",
    "red-zeppelin": "AVL Red Zeppelin",
    "phazite": "Phazite",
}
snare_names = {"3": "beat 3", "24": "beats 2 and 4", "13": "beats 1 and 3"}
comparisons = {
    ("3", "24"): "Beat 3 vs. beats 2 and 4",
    ("13", "24"): "Beats 1 and 3 vs. beats 2 and 4 (equal hits)",
}
group_names = {"acoustic": "Acoustic to acoustic", "electronic": "Acoustic to electronic"}
outcome_names = {"rhythm": "Rhythm closer", "kit": "Kit closer", "tie": "Tie"}
colors = {"rhythm": "#126860", "kit": "#aa4a26", "tie": "#6b7581"}

with Path("trials.csv").open(encoding="utf-8-sig", newline="") as source:
    trials = list(DictReader(source))
for row in trials:
    for column in ("rhythm_score", "kit_score", "margin", "aligned_margin"):
        row[column] = float(row[column])

by_group = {group: [row for row in trials if row["group"] == group] for group in group_names}
counts = {group: Counter(row["outcome"] for row in rows) for group, rows in by_group.items()}
totals = {group: len(rows) for group, rows in by_group.items()}

def show(html):
    display(HTML(html))

def label(snare, kit):
    return f"{kit_names[kit]}, {snare_names[snare]}"

def audio_path(pattern, snare, kit):
    return f"audio/{pattern}_{snare}_{kit}.wav"

def margin_text(row, margin_key="margin", outcome_key="outcome"):
    outcome = row[outcome_key]
    return f'<span class="value {outcome}">{row[margin_key]:+.4f}</span>{outcome_names[outcome]}'

def table(headers, rows):
    heading = "".join(f'<th scope="col">{value}</th>' for value in headers)
    body = "".join(
        f'<tr><th scope="row">{first}</th>' + "".join(f'<td>{value}</td>' for value in rest) + '</tr>'
        for first, *rest in rows
    )
    return f'<div class="table-wrap"><table class="data-table"><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table></div>'

kit_rate = sum(group["kit"] for group in counts.values()) / len(trials)
show(
    f'<p class="lead-finding">MERT favored the same drum kit in {100 * kit_rate:g}% of whole-loop trials.</p>'
    '<p>Each reference loop faces two alternatives. One keeps the snare rhythm and changes the drum kit. '
    'The other keeps the kit and changes the snare rhythm. A positive margin means the same-rhythm loop '
    'was closer in MERT-v1-95M. A negative margin means the same-kit loop was closer.</p>'
)

## Hear both swaps

Listen to the reference and its rhythm change. Then listen to how the reference sounds on another acoustic kit and on the electronic kit.

In [ ]:
#| output: asis

eligible = defaultdict(list)
for row in trials:
    if row["kit"] != "phazite" and {row["snare"], row["other_snare"]} == {"3", "24"}:
        eligible[row["pattern"], row["snare"], row["kit"]].append(row)
opening_pairs = [
    (acoustic, electronic)
    for rows in eligible.values()
    for acoustic in rows if acoustic["group"] == "acoustic"
    for electronic in rows if electronic["group"] == "electronic"
]

def paired_change(pair):
    acoustic, electronic = pair
    return electronic["margin"] - acoustic["margin"]

median_change = median(map(paired_change, opening_pairs))
flips = [pair for pair in opening_pairs if {row["outcome"] for row in pair} == {"rhythm", "kit"}]
acoustic, electronic = min(flips or opening_pairs, key=lambda pair: abs(paired_change(pair) - median_change))
pattern, snare, kit = (acoustic[key] for key in ("pattern", "snare", "kit"))
opening_clips = [
    ("Reference", snare, kit),
    ("Rhythm change", acoustic["other_snare"], kit),
    ("Acoustic kit change", snare, acoustic["other_kit"]),
    ("Electronic kit change", snare, electronic["other_kit"]),
]
def listening_grid(pattern, clips):
    cards = []
    for role, snare, kit in clips:
        description = label(snare, kit)
        cards.append(
            f'<div class="listening-card"><p><strong>{role}</strong><br>{description}</p>'
            f'<audio controls preload="metadata" data-external="1" src="{audio_path(pattern, snare, kit)}" '
            f'aria-label="{role}, {description}"></audio></div>'
        )
    return '<div class="listening-grid">' + ''.join(cards) + '</div>'

example = "The comparison below shows one switch." if flips else "This example is closest to the median margin change."
show(
    f'<p><strong>Backing pattern {pattern[1:]}.</strong> The winner switched between rhythm and kit in '
    f'{len(flips)} of {len(opening_pairs)} eligible paired examples. {example}</p>'
    + listening_grid(pattern, opening_clips)
)
show(table(
    ["Kit change", "Same rhythm", "Same kit", "Whole-loop margin"],
    [
        (kit_names[row["other_kit"]], f'{row["rhythm_score"]:.4f}', f'{row["kit_score"]:.4f}', margin_text(row))
        for row in (acoustic, electronic)
    ],
))

## Results by kit swap

The acoustic-to-electronic group includes trials with a reference from either kit. This section uses whole-loop scores.

In [ ]:
#| output: asis

chart_style = {
    "template": "plotly_white",
    "paper_bgcolor": "#faf9f6",
    "font": {"family": "system-ui, sans-serif", "color": "#212b32", "size": 13},
}

from plotly.offline import get_plotlyjs

show(f'<script type="text/javascript">{get_plotlyjs()}</script>')

def show_chart(fig, title, div_id, post_script=None):
    chart = fig.to_html(
        full_html=False, include_plotlyjs=False, div_id=div_id, post_script=post_script,
        config={"displayModeBar": False, "responsive": True},
    )
    show(f'<div role="group" aria-label="{title}">{chart}</div>')

show(table(
    ["Kit swap", "Rhythm wins", "Kit wins", "Ties", "Trials"],
    [
        (name, *(counts[group][outcome] for outcome in outcome_names), totals[group])
        for group, name in group_names.items()
    ],
))

### How far apart were the scores?

Each dot is one trial. The vertical line marks a tied score. In the equal-hit comparison, each snare part has two hits per bar.

In [ ]:
#| output: asis

def comparison_key(row):
    return next(pair for pair in comparisons if set(pair) == {row["snare"], row["other_snare"]})

margins = make_subplots(
    rows=2, cols=1, shared_xaxes=True, subplot_titles=list(comparisons.values()), vertical_spacing=0.18,
)
for index, pair in enumerate(comparisons, start=1):
    for group, group_name in group_names.items():
        rows = [row for row in by_group[group] if comparison_key(row) == pair]
        margins.add_trace(
            go.Box(
                name=group_name, orientation="h", x=[row["margin"] for row in rows],
                y=[group_name] * len(rows), boxpoints="all", jitter=0.57, pointpos=0,
                marker={"color": "#225d72" if group == "acoustic" else "#775438", "size": 5, "opacity": 0.7},
                line={"width": 1.2}, showlegend=False,
                xhoverformat=".3f", hoverinfo="x", hoverlabel={"namelength": 0},
                text=[
                    f'Pattern {row["pattern"][1:]} | {label(row["snare"], row["kit"])}'
                    f'<br>Swap to {kit_names[row["other_kit"]]}'
                    f'<br>Margin {row["margin"]:+.3f}'
                    for row in rows
                ],
                hovertemplate="%{text}<extra></extra>",
            ),
            row=index, col=1,
        )
    margins.add_vline(x=0, line_width=1, line_dash="dash", line_color="#6b7581", row=index, col=1)
margins.update_layout(**chart_style, height=415, margin={"l": 5, "r": 20, "t": 32, "b": 64})
margins.update_annotations(font_size=14)
margins.update_yaxes(autorange="reversed")
margin_extent = [row["margin"] for row in trials] + [0]
margin_padding = (max(margin_extent) - min(margin_extent)) * 0.04
margins.update_xaxes(range=[min(margin_extent) - margin_padding, max(margin_extent) + margin_padding])
margins.update_xaxes(title_text="Whole-loop margin (same rhythm minus same kit)", row=2, col=1)
# Plotly generates a separate SVG tooltip for each box statistic.
show_chart(margins, "Score margins for both snare comparisons and both kit-swap groups", "margins",
           post_script="""
const chart = document.getElementById('{plot_id}');
chart.on('plotly_hover', () => {
    for (const label of chart.querySelectorAll('.hoverlayer .hovertext')) {
        label.style.display = /^(?:[(])?(?:lower|upper) fence:/i.test(label.textContent) ? 'none' : '';
    }
});
""")

acoustic_median = median(row["margin"] for row in by_group["acoustic"])
electronic_median = median(row["margin"] for row in by_group["electronic"])
show(
    f'<p>The median whole-loop margin was {acoustic_median:+.3f} for acoustic-to-acoustic swaps '
    f'and {electronic_median:+.3f} for acoustic-to-electronic swaps.</p>'
)

## What changes when timing is preserved?

The whole-loop score compares one averaged MERT vector per loop. The aligned score compares the vectors frame by frame at the same time positions and averages those similarities.

In [ ]:
#| output: asis

timing_rows = []
for group, group_name in group_names.items():
    rows = by_group[group]
    aligned = Counter(row["aligned_outcome"] for row in rows)
    changes = Counter(
        (row["outcome"], row["aligned_outcome"]) for row in rows
        if row["aligned_outcome"] != row["outcome"]
    )
    change_note = f"{sum(changes.values())}/{len(rows)}"
    if changes:
        change_note += "<br>" + ", ".join(
            f"{source} to {target}: {count}" for (source, target), count in sorted(changes.items())
        )
    timing_rows.append((
        group_name, *(f"{aligned[outcome]} ({aligned[outcome] / len(rows):.1%})" for outcome in outcome_names),
        change_note,
    ))
show(table(["Kit swap", "Aligned rhythm wins", "Aligned kit wins", "Aligned ties", "Outcome changes"], timing_rows))

timing_flips = [
    row for row in trials
    if row["outcome"] == "kit" and row["aligned_outcome"] == "rhythm"
]
if timing_flips:
    example = max(timing_flips, key=lambda row: row["aligned_margin"])
    show(
        '<h3>Hear a change of winner</h3>'
        f'<p><strong>Backing pattern {example["pattern"][1:]}.</strong> '
        f'The same-kit loop was closer in the whole-loop comparison ({example["margin"]:+.4f}). '
        f'The same-rhythm loop was closer when the frames were aligned ({example["aligned_margin"]:+.4f}).</p>'
        + listening_grid(example["pattern"], [
            ("Reference", example["snare"], example["kit"]),
            ("Rhythm change", example["other_snare"], example["kit"]),
            ("Kit change", example["snare"], example["other_kit"]),
        ])
    )

## Can MERT find the same performance on another kit?

In [ ]:
#| output: asis
from retrieval import read_similarities, retrieval_summary

summary = retrieval_summary(*read_similarities())
show(f"<p>{summary} A random pick would recover the pattern 8.3% of the time: "
     "three of the 36 eligible recordings play it. These searches reuse the same loops.</p>")

[Explore the five nearest loops and the kit-by-kit heatmap](neighbor-exploration.html). Try switching the same reference between all kits and other kits only to see how its neighbors change.

## Browse the trials

Expand a kit pair to see its four reference positions. Candidate numbers point to clips in the same table. Select a numbered clip to play it, then use the audio controls to replay. Scores are rounded for display. Win and tie labels use full precision.

In [ ]:
#| output: asis
from base64 import b64encode

encoded_csv = b64encode(Path("trials.csv").read_bytes()).decode("ascii")
show(f'<a download="trials.csv" href="data:text/csv;base64,{encoded_csv}">Download all {len(trials)} trial scores</a>')

In [ ]:
#| output: asis

def trial_set(pattern, pair, kits, rows):
    positions = [(snare, kit) for kit in kits for snare in pair]
    clip_numbers = {clip: number for number, clip in enumerate(positions, start=1)}
    indexed_rows = {(row["snare"], row["kit"]): row for row in rows}
    display_rows = []
    for (snare, kit), number in clip_numbers.items():
        row = indexed_rows[snare, kit]
        rhythm_number = clip_numbers[snare, row["other_kit"]]
        kit_number = clip_numbers[row["other_snare"], kit]
        button = (
            f'<button type="button" data-clip="{audio_path(pattern, snare, kit)}" aria-pressed="false">'
            f'{number}. {label(snare, kit)}</button>'
        )
        display_rows.append((
            button, f'Clip {rhythm_number}<br>{row["rhythm_score"]:.4f}',
            f'Clip {kit_number}<br>{row["kit_score"]:.4f}',
            margin_text(row), margin_text(row, "aligned_margin", "aligned_outcome"),
        ))
    title = f'{kit_names[kits[0]]} and {kit_names[kits[1]]}'
    headings = ["Reference clip", "Same rhythm", "Same kit", "Whole-loop margin", "Aligned margin"]
    return (
        f'<details class="trial-set"><summary>{title}</summary>'
        + table(headings, display_rows)
        + '<audio controls hidden preload="none"></audio></details>'
    )

trial_sets = defaultdict(list)
for row in trials:
    kits = tuple(kit for kit in kit_names if kit in {row["kit"], row["other_kit"]})
    trial_sets[row["pattern"], comparison_key(row), kits].append(row)

for pattern in sorted({row["pattern"] for row in trials}):
    sections = [f'<details class="trial-set"><summary>Backing pattern {pattern[1:]}</summary>']
    for pair, title in comparisons.items():
        sections.append(f'<h3>{title}</h3>')
        for kits in combinations(kit_names, 2):
            sections.append(trial_set(pattern, pair, kits, trial_sets[pattern, pair, kits]))
    show(''.join(sections) + '</details>')

<script>
document.addEventListener("click", event => {
    const button = event.target.closest("button[data-clip]");
    if (!button) return;
    const section = button.closest(".trial-set");
    section.querySelectorAll("button[data-clip]").forEach(clip => {
        clip.setAttribute("aria-pressed", String(clip === button));
    });
    const player = section.querySelector("audio");
    player.src = button.dataset.clip;
    player.setAttribute("aria-label", button.textContent);
    player.hidden = false;
    player.play();
});
document.addEventListener("play", event => {
    if (!(event.target instanceof HTMLAudioElement)) return;
    document.querySelectorAll("audio").forEach(player => {
        if (player !== event.target && !player.paused) player.pause();
    });
}, true);
</script>

## How the study works

Four kick-and-hi-hat patterns were rendered with three snare parts and four kits. Each clip has two bars at 120 BPM. Within each pattern and snare comparison, both kits and both snare parts take a turn as the reference. The kick and hi-hat stay at the same times for all four reference positions.

MERT-v1-95M processed each loop as mono 24 kHz audio. The whole-loop comparison calculates cosine similarity between the mean final-layer vectors. The margin subtracts the similarity to the same-kit candidate from the similarity to the same-rhythm candidate. The aligned check applies the same subtraction to the average of frame-wise cosine similarities.

The trial totals describe these sounds and patterns. Trials share source clips, and Phazite is the only electronic kit in the study. The kit comparison can reflect the full sound of its kick, snare and hi-hat.

[MERT-v1-95M model card](https://huggingface.co/m-a-p/MERT-v1-95M){target="_blank" rel="noopener"}

In [ ]:
#| output: asis
# Render the study README in place so these instructions also travel with the HTML.
readme = Path("README.md").read_text(encoding="utf-8-sig")
print('<details><summary>Study files and reproduction instructions</summary>\n')
for line in readme[readme.index("## Listen"):].splitlines():
    print(f"**{line[3:]}**" if line.startswith("## ") else line)
print('\n</details>')